In [1]:
import torch
import numpy as np

import time
import quairkit as qkit
from quairkit import Circuit, to_state
from quairkit.database import *
from quairkit.operator import QuasiOperation
from quairkit.qinfo import *
from quairkit import *
from quairkit.loss import *
qkit.set_dtype('complex64')


In [2]:
p0 = torch.nn.Parameter(torch.tensor(2.0))

In [3]:
# Layer ansatz
depth = 10
def f(param: torch.Tensor) -> torch.Tensor:
    param = param.view([2, depth, 2,-1])
    
    cir1 = Circuit(3)
    for i in range(depth):
        cir1.rx(param=param[0, i, 0, :])
        cir1.rz(param=param[0, i, 1, :])
        cir1.cnot([0,1])
        cir1.cnot([1,2])

    cir2= Circuit(3)
    for i in range(depth):
        cir2.rx(param=param[1, i, 0, :])
        cir2.rz(param=param[1, i, 1, :])
        cir2.cnot([0,1])
        cir2.cnot([1,2])
    list_unitary = []
    list_unitary.append(cir1.matrix)
    list_unitary.append(cir2.matrix)
    
    return torch.stack(list_unitary)


In [ ]:
state =  nkron(bell_state(2), zero_state(2))


cir = Circuit(4)
# cir.depolarizing(0.2, 1)
cir.amplitude_damping(0.3, 1)
cir.param_quasi(f, depth*6, p0, system_idx=[1,2,3], probability_param=True)



In [5]:
def loss_fcn(circuit: Circuit) -> torch.Tensor:
    output_state = circuit(state).trace([2,3])
    # cost = 10*(trace_distance(output_state.expec_state(), bell_state(2))) + output_state.probability.abs().sum()
    cost = 1000 * (1 + trace(output_state.expec_state() @ output_state.expec_state()) - 2*(trace(output_state.expec_state() @ bell_state(2)))) + output_state.probability.abs().sum()
    return torch.real(cost)

def overhead_fcn(circuit: Circuit) -> torch.Tensor:
    return circuit(state).probability.abs().sum()

def distance_fcn(circuit: Circuit) -> torch.Tensor:
    output_state = circuit(state).trace([2,3])
    return (1 + trace(output_state.expec_state() @ output_state.expec_state()) - 2*(trace(output_state.expec_state() @ bell_state(2))))
    # return trace_distance(circuit(state).trace([2,3]).expec_state(), bell_state(2)) 

In [6]:
LR, NUM_ITR = 0.05, 800
loss_list, time_list = [], []


In [7]:
opt = torch.optim.Adam(lr=LR, params=cir.parameters()) # cir is a Circuit type
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5) # activate scheduler

for itr in range(NUM_ITR):
    start_time = time.time()
    opt.zero_grad()

    loss = loss_fcn(cir) # compute loss

    loss.backward()
    opt.step()
    scheduler.step(loss) # activate scheduler

    loss = loss.item()
    loss_list.append(loss)
    time_list.append(time.time() - start_time)

    if itr % (NUM_ITR // 20) == 0 or itr == NUM_ITR - 1:
        overhead = overhead_fcn(cir).item() # compute fidelity
        distance = distance_fcn(cir).item()
        print(f"iter: {str(itr).zfill(len(str(NUM_ITR)))}, " +
              f"loss: {loss:.8f}, overhead: {overhead:.8f}, " +
              f"distance: {distance:.7f}, avg_time: {np.mean(time_list):.4f}s")
        time_list = []

c:\Users\bench\.conda\envs\quairkit\lib\site-packages\torch\optim\lr_scheduler.py:1340: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:836.)
  current = float(metrics)


iter: 000, loss: 721.37921143, overhead: 2.90000010, distance: 0.2741771-0.0000000j, avg_time: 0.0781s
iter: 040, loss: 2.64860153, overhead: 2.13470554, distance: 0.0003486+0.0000000j, avg_time: 0.0504s
iter: 080, loss: 2.02064085, overhead: 2.00576949, distance: 0.0000092-0.0000000j, avg_time: 0.0558s
iter: 120, loss: 1.94186068, overhead: 1.93676376, distance: 0.0000039+0.0000000j, avg_time: 0.0482s
iter: 160, loss: 1.89867544, overhead: 1.89348340, distance: 0.0000043+0.0000000j, avg_time: 0.0505s
iter: 200, loss: 1.87591076, overhead: 1.87098980, distance: 0.0000045+0.0000000j, avg_time: 0.0579s
iter: 240, loss: 1.86481786, overhead: 1.85997772, distance: 0.0000045+0.0000000j, avg_time: 0.0545s
iter: 280, loss: 1.85932612, overhead: 1.85445905, distance: 0.0000046-0.0000000j, avg_time: 0.0526s
iter: 320, loss: 1.85638642, overhead: 1.85156417, distance: 0.0000048+0.0000000j, avg_time: 0.0506s
iter: 360, loss: 1.85468912, overhead: 1.85001016, distance: 0.0000046-0.0000000j, avg_ti